# Module 8: Statistical inference

**Goal:** say *how much you can trust* what a sample shows. Your data are one sample of what could have been collected;
inference quantifies how much results would wobble from sample to sample.

**You will learn:** sampling variation · standard error · confidence intervals (formula **and** bootstrap) · hypothesis tests
(permutation, *t*-test, correlation, chi-square) · effect sizes · errors and power · multiple comparisons · how to report results honestly

**Maths in this module:** standard error $\sigma/\sqrt n$ · $\bar x\pm t^*\,\text{SE}$ · the p-value · Cohen's $d$ · Bonferroni · $\chi^2$

> 🧭 Big idea: **an estimate without a statement of uncertainty is just a guess.** Every number you report
> (a mean, a difference, a correlation) should travel with an interval.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")
rng = np.random.default_rng(8)

students = pd.read_csv(DATA / "students.csv")
scores = students["exam_score"]

## 1. One sample, many possible samples

We care about a **population** (say, *all* students at a university) but only see a **sample**. A **statistic** (the sample mean $\bar x$) is an
estimate of a **parameter** (the population mean $\mu$). Collect another sample and you'd get a different $\bar x$.

Let's make a pretend population where we *know* the truth, then draw many samples of 25 and watch the sample means vary:

In [ ]:
true_mu, true_sigma, n = 68, 10.5, 25
sample_means = rng.normal(true_mu, true_sigma, size=(10_000, n)).mean(axis=1)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(sample_means, bins=50, edgecolor="white")
ax.axvline(true_mu, color="tab:red", linestyle="--", label="true mean")
ax.set_xlabel("Sample mean (n=25)"); ax.set_ylabel("Number of samples"); ax.legend()
plt.show()

print("sd of sample means:", sample_means.std().round(3), "| theory σ/√n =", round(true_sigma / math.sqrt(n), 3))

The spread of the sample means is the **standard error** (SE). By the CLT (Module 7):

$$\text{SE}(\bar x)=\frac{\sigma}{\sqrt n}\quad\approx\quad \frac{s}{\sqrt n}\ \text{(using the sample sd } s\text{)}$$

Quadrupling the sample size only **halves** the uncertainty.

## 2. Confidence intervals

A **95% confidence interval** for a mean is

$$\bar x\;\pm\;t^*\cdot\text{SE},\qquad t^*=\text{the 97.5th percentile of the } t \text{ distribution with } n-1 \text{ degrees of freedom}$$

(For large $n$, $t^*\approx1.96$.) Applied to the exam scores:

In [ ]:
n = len(scores)
xbar, s = scores.mean(), scores.std()
se = s / math.sqrt(n)
t_star = stats.t.ppf(0.975, df=n - 1)

low, high = xbar - t_star * se, xbar + t_star * se
print(f"mean {xbar:.2f}, SE {se:.3f}, t* {t_star:.3f}")
print(f"95% CI: ({low:.2f}, {high:.2f})")
print("scipy agrees:", stats.t.interval(0.95, df=n - 1, loc=xbar, scale=se))

### What does "95%" mean?

**Not** "there's a 95% chance the true mean is in *this* interval." The truth is fixed; the interval is what's random.
The correct reading: *if we repeated the whole study many times, 95% of the intervals built this way would contain the true mean.*
Let's watch it happen:

In [ ]:
covered = 0
trials = 2_000
for _ in range(trials):
    sample = rng.normal(true_mu, true_sigma, size=25)
    se_i = sample.std(ddof=1) / 5
    half = stats.t.ppf(0.975, 24) * se_i
    covered += (sample.mean() - half <= true_mu <= sample.mean() + half)
print(f"{covered / trials:.1%} of {trials} intervals contained the true mean")

## 3. The bootstrap: a CI for (almost) anything

The formula above works for means. What about a **median**, a correlation or any odd statistic? The **bootstrap** uses a clever trick:
treat your sample as a stand-in for the population and **resample it with replacement**, many times, recomputing the statistic each time.
The spread of those results approximates the sampling variation.

In [ ]:
def bootstrap_ci(values, stat=np.mean, n_boot=5000, confidence=0.95, rng=rng):
    values = np.asarray(values)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))     # resample row indices with replacement
    boot_stats = np.apply_along_axis(stat, 1, values[idx])
    alpha = (1 - confidence) / 2
    return np.quantile(boot_stats, [alpha, 1 - alpha])

print("mean   95% CI:", bootstrap_ci(scores).round(2), "  (compare t-interval above)")
print("median 95% CI:", bootstrap_ci(scores, stat=np.median).round(2))

The bootstrap mean interval matches the formula, and now we also have one for the median, **with no extra maths**.
Use it for statistics without a tidy formula, but remember it can't fix a biased or tiny sample.

## 4. Hypothesis tests: is it just chance?

**Question:** do students with a part-time job score differently from those without?

In [ ]:
jobs = students.groupby("part_time_job")["exam_score"].agg(["count", "mean", "std"]).round(2)
print(jobs)
yes = students.loc[students["part_time_job"] == "yes", "exam_score"].to_numpy()
no = students.loc[students["part_time_job"] == "no", "exam_score"].to_numpy()
observed = no.mean() - yes.mean()
print("observed difference (no job - job):", round(observed, 2))

The logic of a test:

1. **Null hypothesis $H_0$:** "there is no real difference; the gap came from chance in who ended up in which group."
2. Compute a **test statistic** (here, the difference in means).
3. Ask: **if $H_0$ were true, how often would chance alone produce a difference at least this big?** That probability is the **p-value**.
4. Small p-value → the data are hard to explain by chance alone. Compare with a threshold $\alpha$ chosen beforehand (often 0.05).

### A permutation test: the logic made visible
If job status didn't matter, the labels "yes/no" are interchangeable. So **shuffle** them and see what differences arise by luck:

In [ ]:
pooled = np.concatenate([yes, no])
diffs = np.empty(10_000)
for i in range(len(diffs)):
    rng.shuffle(pooled)
    diffs[i] = pooled[len(yes):].mean() - pooled[:len(yes)].mean()

p_perm = (np.abs(diffs) >= abs(observed)).mean()          # two-sided: extreme in either direction

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(diffs, bins=60, edgecolor="white")
ax.axvline(observed, color="tab:red"); ax.axvline(-observed, color="tab:red")
ax.set_xlabel("Difference in means under 'no real difference'"); ax.set_ylabel("Shuffles")
ax.set_title(f"Permutation test: p = {p_perm:.3f}")
plt.show()

### The classical version: Welch's *t*-test
$$t=\frac{\bar x_1-\bar x_2}{\sqrt{s_1^2/n_1+s_2^2/n_2}}$$
The permutation and *t* approaches agree closely; the *t*-test is a formula, the permutation test is a simulation.

In [ ]:
res = stats.ttest_ind(no, yes, equal_var=False)            # Welch: doesn't assume equal spreads
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.4f}")
print("95% CI for the difference:", np.round(res.confidence_interval(0.95), 2))

### Reading the result honestly

The p-value is about 0.07, just above 0.05. Two *wrong* conclusions tempt us:

* ❌ "p > 0.05, so job status has **no effect**." *Absence of evidence is not evidence of absence.*
* ❌ "p = 0.07 is basically significant, so it's real." Thresholds are conventions, not cliffs.

The interval tells the fuller story: the difference could be anywhere from about **0 to 6 points**. The data are *compatible with no difference
and with a sizeable one.* That is inconclusive, and a larger sample would help. **Always report the interval, not just the verdict.**

**What a p-value is *not*:** it's not the probability $H_0$ is true, not the probability the result is due to chance, and not the size or importance of an effect.

## 5. Effect size: statistically significant ≠ important

**Cohen's $d$** expresses a difference in standard-deviation units:
$$d=\frac{\bar x_1-\bar x_2}{s_p},\qquad s_p=\sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}}$$
Rough guide: 0.2 small · 0.5 medium · 0.8 large (context matters!). With a huge sample even a trivial difference becomes "significant":

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_var = ((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2)
    return (np.mean(a) - np.mean(b)) / math.sqrt(pooled_var)

print("job vs no job: d =", round(cohens_d(no, yes), 2), "(small-to-medium, but uncertain)")

big_a = rng.normal(100.0, 15, 200_000)
big_b = rng.normal(100.3, 15, 200_000)                      # a 0.3-point difference on a 15-point spread
print("huge sample: p =", stats.ttest_ind(big_a, big_b).pvalue, "| d =", round(cohens_d(big_b, big_a), 3))

## 6. Errors and power

| | $H_0$ true (no effect) | $H_0$ false (real effect) |
|---|---|---|
| **reject $H_0$** | **Type I error** (false alarm), probability $\alpha$ | correct, probability = **power** |
| **don't reject** | correct | **Type II error** (missed effect), probability $\beta=1-\text{power}$ |

**Power** is the chance of detecting a real effect. It depends on effect size, sample size and $\alpha$. Let's measure it by simulation: a *real* difference of $d=0.5$ with 30 people per group.

In [ ]:
def simulated_power(d, n_per_group, sims=2000, alpha=0.05):
    hits = 0
    for _ in range(sims):
        g1 = rng.normal(0, 1, n_per_group)
        g2 = rng.normal(d, 1, n_per_group)
        hits += stats.ttest_ind(g1, g2).pvalue < alpha
    return hits / sims

for n in [15, 30, 64, 100]:
    print(f"n={n:3d} per group -> power {simulated_power(0.5, n):.0%}")

With 30 per group you'd **miss a real medium-sized effect more than half the time**. Rule of thumb: ~64 per group gives 80% power for $d=0.5$.
**Plan your sample size before collecting data**, and treat "not significant" in a small study as *inconclusive*.

## 7. Many tests, many false alarms

Run 20 tests on pure noise at $\alpha=0.05$ and you expect about **one** "significant" result by luck.

In [ ]:
p_values = [stats.ttest_ind(rng.normal(size=20), rng.normal(size=20)).pvalue for _ in range(100)]
print("tests on pure noise:", len(p_values))
print("'significant' at 0.05:", sum(p < 0.05 for p in p_values))
print("after Bonferroni (α/100 =", 0.05 / 100, "):", sum(p < 0.05 / 100 for p in p_values))

This is the root of **p-hacking** ("test many things, report the one that worked") and of many irreproducible findings. Defences:

* **Decide your questions before looking** (write them in your project proposal!)
* Report **how many** comparisons you made
* Adjust: **Bonferroni** uses $\alpha/m$ for $m$ tests (simple but conservative)
* Confirm surprising results on **new data**

## 8. Other everyday tests

**Correlation:** is a relationship stronger than chance? (And give a bootstrap CI for $r$.)

In [ ]:
r = stats.pearsonr(students["study_hours_week"], scores)
print(f"r = {r.statistic:.2f}, p = {r.pvalue:.1e}")

pairs = students[["study_hours_week", "exam_score"]].to_numpy()
idx = rng.integers(0, len(pairs), size=(5000, len(pairs)))
boot_r = [np.corrcoef(pairs[i, 0], pairs[i, 1])[0, 1] for i in idx]
print("bootstrap 95% CI for r:", np.quantile(boot_r, [0.025, 0.975]).round(2))

**Two categories:** the **chi-square test of independence** compares the counts you see with the counts you'd expect if the variables were unrelated:
$$\chi^2=\sum\frac{(\text{observed}-\text{expected})^2}{\text{expected}}$$

In [ ]:
table = pd.crosstab(students["major"], students["part_time_job"])
print(table)
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.2f}  -> no evidence that job status depends on major")

**A proportion** (e.g. share of students scoring 50+):
$$\hat p\pm1.96\sqrt{\frac{\hat p(1-\hat p)}{n}}$$

In [ ]:
p_hat = (scores >= 50).mean()
half = 1.96 * math.sqrt(p_hat * (1 - p_hat) / len(scores))
print(f"{p_hat:.1%} passed, 95% CI ({p_hat - half:.1%}, {p_hat + half:.1%})")

## 9. Correlation, causation and study design

Tests tell you whether a pattern is *unlikely to be chance*, not **why** it exists. If students who study more score higher,
maybe studying helps, maybe motivated students do both, maybe both depend on free time. Only a **randomised experiment** (assign who gets the treatment
by lottery) breaks those links. With observational data, say "is associated with", and discuss alternative explanations.

## 10. A reporting recipe

For every result in your project, give: **what** you compared · the **estimate** (with units) · its **interval** · the **p-value** (if you tested) · an **effect size** · the **sample size** · **caveats**.

> *Students with a part-time job scored on average 2.9 points lower than those without (66.0 vs 68.9; Welch 95% CI for the difference −0.3 to 6.1; p = 0.07; Cohen's d ≈ 0.28; n = 68 vs 132).
> The data are compatible with no difference and with a moderate one, so this is suggestive but inconclusive. The study is observational, so the gap is not evidence that jobs cause lower scores.*

## Recap

* Sampling variation is unavoidable: **SE = s/√n**; the CLT makes it predictable.
* **Confidence intervals** (formula or bootstrap) express uncertainty. **Permutation / t-tests** ask "could chance explain this?"
* A p-value is *not* the probability the null is true; a large p-value is *not* "no effect".
* Report **effect size and interval**, plan **power**, and beware **multiple comparisons**.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** (`analysis.py`) in the README.